## 1. Datapræparation & Label Mapping

In [3]:
import pandas as pd
import re
from rich.progress import track
import nltk
from matplotlib import pyplot as plt

from transformers import RobertaTokenizerFast
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
from transformers import RobertaForSequenceClassification
import time
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup

In [4]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/My Drive/GDS_group_project_main/Group_project/995,000_rows.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
raw_data_subset = pd.read_csv(path, dtype = 'string').head(20000)

### Label mapping

In [6]:
fake_news = {'fake', 'satire', 'bias', 'conspiracy', 'junksci', 'hate', 'rumor'}
reliable_news = {'reliable', 'unreliable', 'political', 'clickbait'}

mapping = {label: 0 for label in fake_news}
mapping.update({label: 1 for label in reliable_news})

#creating binary label cloumn from the 'type' column
raw_data_subset['label'] = raw_data_subset['type'].map(mapping)

# 4. Fjern rækker der ikke passede ind i dine kategorier (dem der blev NaN)
before_count = len(raw_data_subset)
raw_data_subset = raw_data_subset.dropna(subset=['label'])
after_count = len(raw_data_subset)

print(f"Mapping done. Removed {before_count - after_count} rows with invalid label.")

# Konverter label til integer (så det ikke er kommatal)
raw_data_subset['label'] = raw_data_subset['label'].astype(int)

Mapping done. Removed 1755 rows with invalid label.


### Cleaning data (light clean)

In [7]:
EMAIL_REGEX = re.compile(r'\\w+\\w+@\\w+\\.\\w+')
URL_REGEX1 = re.compile(r'https?:((//)|(\\\\))+[^, ]+')
URL_REGEX2 = re.compile(r'www\\.[^, ]+')
DATE_REGEX = re.compile(r'<NUM>-<NUM>-<NUM>.?<NUM>:<NUM>:<NUM>.?<NUM>')

def clean_string(text):
    text = str(text)

    #Handling white space
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    #Subbing with our regex's:
    text = EMAIL_REGEX.sub('<EMAIL>', text)
    text = URL_REGEX1.sub('<URL>', text)
    text = URL_REGEX2.sub('<URL>', text)

    return text

def clean_df(df):

    print("Cleaning content...")
    df['clean'] = df['content'].apply(clean_string)

    return df

cleaned_data = clean_df(raw_data_subset)
cleaned_data.head()


Cleaning content...


,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary,source,label,clean
0,732,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,Plus one article on Google Plus (Thanks to Al...,2017-11-27T01:14:42.983556,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,Iran News Round Up,<NA>,<NA>,"['National Review', 'National Review Online', ...",<NA>,<NA>,<NA>,<NA>,1,Plus one article on Google Plus (Thanks to Ali...
1,1348,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,The Cost Of The Best Senate Banking Committee ...,2017-11-27T01:14:08.7454,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,The Cost Of The Best Senate Banking Committee ...,<NA>,<NA>,[''],<NA>,<NA>,<NA>,<NA>,0,The Cost Of The Best Senate Banking Committee ...
2,7119,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,Man Awoken From 27-Year Coma Commits Suicide A...,2017-11-27T01:14:21.395055,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,Man Awoken From 27-Year Coma Commits Suicide A...,<NA>,<NA>,[''],<NA>,<NA>,<NA>,<NA>,0,Man Awoken From 27-Year Coma Commits Suicide A...
3,1518,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,WHEN Julia Geist was asked to draw a picture o...,2018-02-11 00:46:42.632962,2018-02-11 00:14:20.346838,2018-02-11 00:14:20.346871,Opening a Gateway for Girls to Enter the Compu...,<NA>,<NA>,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,<NA>,<NA>,nytimes,1,WHEN Julia Geist was asked to draw a picture o...
4,9345,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,– 100 Compiled Studies on Vaccine Dangers (Act...,2017-11-10T11:18:44.524042,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,100 Compiled Studies on Vaccine Dangers – Infi...,<NA>,<NA>,[''],<NA>,"Lymphoma, Hepatitis B, Immune System, Health, ...",<NA>,<NA>,0,– 100 Compiled Studies on Vaccine Dangers (Act...


## 2. Tokenization

In [8]:
# 1. Initializing tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')

# 2. going from DataFrame to Dataset (from hugging face)
dataset = Dataset.from_pandas(raw_data_subset[['clean', 'label']])

# 3. Tokenising in batches
def tokenize_function(examples):
    return tokenizer(
        examples['clean'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# 4. Kør tokeniseringen over hele datasættet (lynhurtigt)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 5. Gør det klar til PyTorch
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/18245 [00:00<?, ? examples/s]

## 3. Data Loading (Batches)

In [9]:
# 1. Split into 80% training and 20% for the rest (validation + test)
# Using a seed (random_state) ensures the split is reproducible during testing
train_rem_split = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

train_dataset = train_rem_split['train']
remaining_dataset = train_rem_split['test']

# 2. Split the remaining 20% into two equal halves (10% Validation, 10% Test)
val_test_split = remaining_dataset.train_test_split(test_size=0.5, seed=42)

val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

# Print sizes to confirm the distribution
print(f"Training rows (80%):    {len(train_dataset)}")
print(f"Validation rows (10%):  {len(val_dataset)}")
print(f"Test rows (10%):        {len(test_dataset)}")

# 3. Define Batch Size
# Start with 16. If you encounter "Out of Memory" errors, reduce this to 8.
BATCH_SIZE = 8

# 4. Create DataLoaders
# shuffle=True is CRITICAL for the training set so the model doesn't learn the order
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# For validation and testing, order doesn't matter, so shuffle is False
val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Training rows (80%):    14596
Validation rows (10%):  1824
Test rows (10%):        1825


## 4. Model Definition

In [10]:
# 1. Checking if a GPU is available and set the device accordingly
# Training on a GPU (cuda) is significantly faster than on a CPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

# 2. Initialize the RoBERTa model for sequence classification
# num_labels=2 because we are doing binary classification (Fake vs. Reliable)
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    problem_type="single_label_classification",
    num_labels=2
)

# 3. Move the model to the selected device (GPU or CPU)
model.to(device)

print("Model successfully loaded and moved to device.")

Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model successfully loaded and moved to device.


## 5. Training Loop (Læringsfasen)

In [11]:
# Move model to device (safety check)
model.to(device)

# Set the number of epochs
epochs = 3

# Define the optimizer and learning rate scheduler BEFORE the training loop
optimizer = optim.AdamW(model.parameters(), lr=5e-6, eps=1e-8) # Using AdamW optimizer

# Define a learning rate scheduler
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0, # Default value
    num_training_steps=total_steps
)

print("Starting training...")

for epoch in range(epochs):
    print(f"\n======== Epoch {epoch + 1} / {epochs} ========")

    # 1. Put the model into training mode
    model.train()

    total_train_loss = 0
    t0 = time.time()

    # Iterate over batches from the training dataloader
    for step, batch in enumerate(track(train_dataloader, description=f"Epoch {epoch + 1}")):

        # Progress update every 40 batches - (Removed, as rich.progress.track now handles this)
        #if step % 40 == 0 and not step == 0:
        #    elapsed = time.time() - t0
        #    print(f"  Batch {step} of {len(train_dataloader)}. Elapsed: {elapsed:.2f}s")

        # Push batch to GPU
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['label'].to(device).long() # FIX: Cast labels to long

        # Clear previously calculated gradients
        model.zero_grad()

        # Perform forward pass
        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask,
                        labels=b_labels)

        loss = outputs.loss
        total_train_loss += loss.item()

        # Perform backward pass to calculate gradients
        loss.backward()

        # Clip the norm of the gradients to 1.0 to prevent "exploding gradients"
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Update parameters and take a step using the computed gradient
        optimizer.step()

        # Update the learning rate
        scheduler.step()

    # Calculate average loss over all batches in the epoch
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"\n  Average training loss: {avg_train_loss:.2f}")
    print(f"  Training epoch took: {time.time() - t0:.2f}s")

print("\nTraining complete!")

Output()

Starting training...

======== Epoch 1 / 3 ========


Output()


  Average training loss: 0.55
  Training epoch took: 1511.45s

======== Epoch 2 / 3 ========


Output()


  Average training loss: 0.42
  Training epoch took: 1517.75s

======== Epoch 3 / 3 ========



  Average training loss: 0.38
  Training epoch took: 1517.38s

Training complete!


### Loading the saved model weights

In [ ]:
from transformers import RobertaForSequenceClassification
import torch

# 1. Define the path where the weights are saved
load_weights_path = "/content/drive/My Drive/GDS_group_project_main/Group_project/custom_weight_malte.pth"

# 2. Initialize a new model with the same architecture
# Make sure num_labels matches your training setup (2 for binary classification)
loaded_model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    problem_type="single_label_classification",
    num_labels=2
)

# 3. Load the saved state dictionary into the new model
loaded_model.load_state_dict(torch.load(load_weights_path))

# 4. Move the model to the appropriate device (GPU if available, else CPU)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
loaded_model.to(device)

# 5. Set the model to evaluation mode if you plan to use it for inference
# If you plan to continue training, keep it in .train() mode
loaded_model.eval()

print("Model weights loaded successfully!")

In [12]:
save_weights = "/content/drive/My Drive/GDS_group_project_main/Group_project/custom_weight_malte.pth"
torch.save(model.state_dict(), save_weights)

## 6. Validering & Evaluering

In [13]:
path2 = "/content/drive/My Drive/GDS_group_project_main/Group_project/995,000_rows.csv"

raw_data2 = pd.read_csv(path2, dtype = 'string').drop(raw_data_subset.index)

raw_vdata = raw_data2.head(10000)

#creating binary label cloumn from the 'type' column
raw_vdata['label'] = raw_vdata['type'].map(mapping)

# 4. Fjern rækker der ikke passede ind i dine kategorier (dem der blev NaN)
raw_vdata = raw_vdata.dropna(subset=['label'])

# Konverter label til integer (så det ikke er kommatal)
raw_vdata['label'] = raw_vdata['label'].astype(int)

cleaned_vdata = clean_df(raw_vdata)

/tmp/ipykernel_34449/4175770813.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_vdata['label'] = raw_vdata['type'].map(mapping)


Cleaning content...


In [14]:
# 1. Tokeniser valideringsdataen
# Brug den samme tokeniseringsfunktion som for træningsdataen
v_dataset = Dataset.from_pandas(cleaned_vdata[['clean', 'label']])
tokenized_vdatasets = v_dataset.map(tokenize_function, batched=True)

# 2. Gør det klar til PyTorch
tokenized_vdatasets.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# 3. Opret DataLoader for valideringssættet
val_dataloader_new = DataLoader(
    tokenized_vdatasets,
    batch_size=BATCH_SIZE,
    shuffle=False # Ingen grund til at shuffle valideringsdata
)

print(f"Validation rows (new): {len(tokenized_vdatasets)}")

Map:   0%|          | 0/7476 [00:00<?, ? examples/s]

Validation rows (new): 7476


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Put the model in evaluation mode
model.eval()

total_eval_loss = 0
predictions = []
true_labels = []

print("Running evaluation on the new validation set...")

for batch in track(val_dataloader_new, description="Evaluating"): # Brug den nye dataloader
    b_input_ids = batch['input_ids'].to(device)
    b_input_mask = batch['attention_mask'].to(device)
    b_labels = batch['label'].to(device).long()

    # Deaktiver gradientberegninger for at spare hukommelse og fremskynde inferens
    with torch.no_grad():
        outputs = model(
            b_input_ids,
            token_type_ids=None,
            attention_mask=b_input_mask,
            labels=b_labels
        )

    loss = outputs.loss
    logits = outputs.logits

    total_eval_loss += loss.item()

    # Flyt logits og labels til CPU for beregning af metrics
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    # Gem forudsigelser og sande labels
    predictions.extend(logits.argmax(axis=1))
    true_labels.extend(label_ids)

# Beregn gennemsnitligt tab
avg_eval_loss = total_eval_loss / len(val_dataloader_new)
print(f"\n  Average validation loss: {avg_eval_loss:.2f}")

# Beregn metrics
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')

print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")

print("Evaluation complete!")

Output()

Running evaluation on the new validation set...



  Average validation loss: 0.40
  Accuracy: 0.8192
  Precision: 0.7802
  Recall: 0.8475
  F1-Score: 0.8125
Evaluation complete!
